## inference for variances 

In [2]:
# _rd_00.py 
# 분산 추론. 해보자. 진행 중. 
# 분산의 분산, 카이제곱 이런 것들. K통계량. 
# var() -> ndarray 디폴트 0, dataframe 디폴트 1 <<== 미세한 차이 발생 이유. 

# 분산비는 별도.
# raw data - 연봉, 경력 기간, 성별.
# _rd_00.py ==> 이건 벤치마크용.


import os
import numpy as np                          # numpy 라이브러리 전체. 
import pandas as pd                         # pandas 라이브러리 전체, 시계열(기본) 여기 있네. 
import scipy as sci 
from scipy import stats, optimize, linalg   # scipy 라이브러리 하부 모듈. 필요한 통계 모듈만.
import statsmodels.api as sm                # 방대한 모델이라 개발자들이 많이 쓰는 모델 묶음.
from statsmodels.tsa import stattools       # time series analysis 
from statsmodels.tsa.arima.model import ARIMA # 이거 가능하다 이거지. 이거 대문자네. 꼭. 소문자 못읽어.
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportion_confint, proportions_ztest
from statsmodels.stats.weightstats import DescrStatsW, ztest
from scipy.stats import ttest_1samp

pd.options.display.float_format = '{:.4f}'.format
np.set_printoptions(suppress=True, precision=4)

In [3]:
# 1. 데이터 준비. 읽기. 생성. 
# 데이터 파일 읽기. github.com
dat_url = 'https://github.com/bahn28/gamja/blob/main/cs_nns_gndr_hgt.csv?raw=true'
df_dat = pd.read_csv(dat_url) 
df_dat.head() 

gn = df_dat['gender'].to_numpy()   # 이렇게 하면 배열로 전환. dataframe -> NumPy 배열
hgt = df_dat['ht'].to_numpy()
gndr = (gn == 1).astype(int)    # gn = 1, 2; gndr = 1, 0. 1/0으로 전환. 

df_dat['gndr'] = gndr               #   이것은 위에 1/0 자료를 추가. 성별임.

hgt_m = hgt[gndr == 1]  # 성별 1, m 그룹 키. 그룹별 자료 분리, 배열은 hgt 하나임
hgt_f = hgt[gndr == 0]  #     0, f. 키 그룹별 자료 분리

n_all = len(df_dat) 
n_m = len(hgt_m)
n_f = len(hgt_f)

\begin{align}
K & = (n-1) { \hat\sigma^2 \over \sigma^2} \\
 & \sim \chi^2_{n-1}
\end{align}

$ 100\times (1-\alpha)\% $ 신뢰구간
\begin{align}
{ (n-1)\hat\sigma^2 \over k_2 } < \sigma^2 < { (n-1)\hat\sigma^2 \over k_2 } 
\end{align}

$ P(K < k_1 ) = P(K > k_2) = {\alpha \over 2}$

In [4]:

# 분포 임계치, 양방향, 우측값. 적당한 위치 모색.
alpha = 0.05                          # significance level, two side.
#    kcv_r = stats.chi2.ppf(1-alpha/2, df_all ) # critical value on the right, two-side 
#    kcv_l = stats.chi2.ppf(alpha/2, df_all ) # critical value on the left,
#    kcv_r1 = stats.chi2.ppf(1-alpha, df_all ) # critical value on the right, one-side  
#    zcv_r = stats.norm.ppf( 1- alpha/2 )  # critical value on the right, two side
#    tcv_r = stats.t.ppf(1 - alpha / 2, df= n_tall)

# 3. 분산 추정치: 전체, 성별, 표준오차
# 추정치: 분산, overall
# overall 
sig2hat = hgt.var(ddof=1) 
dof_all = n_all - 1 
k_2a = stats.chi2.ppf(1-alpha/2, dof_all ) # critical value on the right, two-side 
k_1a = stats.chi2.ppf(alpha/2, dof_all )   # critical value on the left,

#    se_muhat = hgt.std() / np.sqrt( n_all )

# 그룹별( m - f)
sig2hat_m = hgt_m.var(ddof=1) 
dof_m = n_m - 1 
k_2m = stats.chi2.ppf(1-alpha/2, dof_m ) # critical value on the right, two-side 
k_1m = stats.chi2.ppf(alpha/2, dof_m ) # critical value on the left,

sig2hat_f = hgt_f.var(ddof=1) 
dof_f = n_f - 1 
k_2f = stats.chi2.ppf(1-alpha/2, dof_f ) # critical value on the right, two-side 
k_1f = stats.chi2.ppf(alpha/2, dof_f ) # critical value on the left,

# 신뢰구간, right, left, 
# overall 
ci_r = dof_all * sig2hat / k_1a 
ci_l = dof_all * sig2hat / k_2a 
print( "         sig2hat,     confidence interval ")
print("overall", sig2hat,  ci_l, ci_r ) 

# 그룹별 ( m - f )
ci_rm = dof_m * sig2hat_m / k_1m 
ci_lm = dof_m * sig2hat_m / k_2m 
#print( "         sig2hat,     confidence interval ")
print("  male ", sig2hat_m,  ci_lm, ci_rm ) 

ci_rf = dof_f * sig2hat_f / k_1f 
ci_lf = dof_f * sig2hat_f / k_2f 
#print( "         sig2hat,     confidence interval ")
print(" female", sig2hat_f,  ci_lf, ci_rf ) 



         sig2hat,     confidence interval 
overall 74.11394543612194 72.68697990995841 75.58354799037583
  male  39.857504083856455 38.714336522239876 41.052309041069776
 female 32.58234286349084 31.745058945023196 33.45335426684616


\begin{align*}
H_0 &: \sigma^2 = \sigma_0^2 \\
H_A &: \sigma^2 \ne \sigma_0^2
\end{align*}

$$ K_0 = (n-1){ \hat \sigma^2 \over \sigma_0^2 }$$

reject $ H_0 $ if $ K_0 > k_2 \text{ or } K_0 < k_1 $

In [6]:

# 가설검정. 전체, 톨 
# 검정통계치, pvalue 

# 귀무가설 H0: mu = mu0
sig2_zero = 60

k_0 =  dof_all * sig2hat / sig2_zero             # overall 
k_0m = dof_m * sig2hat_m / sig2_zero             # male
k_0f = dof_f * sig2hat_f / sig2_zero             # female

yn_h0 = " 'reject h0' " if (k_0 > k_2a or k_0 < k_1a)  else " 'fail to reject h0' "   # 이건 되는 군. 
res = int(k_0 > k_2a or k_0 < k_1a )  # 이것은 기각 성공 1, 기각 실패 0
#    pval = 2*( 1 - stats.chi2.cdf( k_0, dof_all ) ) 
a = stats.chi2.cdf( k_0, dof_all ) 
b = stats.chi2.sf( k_0, dof_all )    # 1- cdf 
pval = 2 * min(a, b)

print("  H0: sigma2 = ", sig2_zero )
print("            K0,  임계치들,  검정결과,  p값  ")
print(" 전체   : ", k_0 , k_1a, k_2a, yn_h0, pval ) 

# ***** 여기는 gender = 1, male 
yn_h0m = " 'reject h0' " if (k_0m > k_2m or k_0m < k_1m )  else " 'fail to reject h0' "   # 이건 되는 군. 
resm = int(k_0m > k_2m or k_0m < k_1m)  # 이것은 기각 성공 1, 기각 실패 0
a = stats.chi2.cdf( k_0m, dof_m ) 
b = stats.chi2.sf( k_0m, dof_m )    # 1- cdf 
pvalm = 2 * min(a, b)
#pvalm = 2*( 1 - stats.chi2.cdf( k_0m, dof_m ) )
print(" 전체m  : ", k_0m , k_1m, k_2m, yn_h0m, pvalm ) 

 
 # ***** 여기는 gender = 0, female 
yn_h0f = " 'reject h0' " if (k_0f > k_2f or k_0f < k_1f )  else " 'fail to reject h0' "   # 이건 되는 군. 
resf = int(k_0f > k_2f or k_0f < k_1f )  # 이것은 기각 성공 1, 기각 실패 0
a = stats.chi2.cdf( k_0f, dof_f ) 
b = stats.chi2.sf( k_0f, dof_f )    # 1- cdf 
pvalf = 2 * min(a, b)
#pvalf = 2*( 1 - stats.chi2.cdf( k_0f, dof_f ) )
print(" 전체f  : ", k_0f , k_1f, k_2f, yn_h0f, pvalf ) 


  H0: sigma2 =  60
            K0,  임계치들,  검정결과,  p값  
 전체   :  24861.522996547104 19735.662316126858 20522.12626856517  'reject h0'  5.654213790690959e-107
 전체m  :  5938.103816759881 8678.83530373785 9202.953247072188  'reject h0'  1.6575690551418998e-144
 전체f  :  6074.977826897867 10895.728622797842 11482.059940260908  'reject h0'  0.0
